<a href="https://colab.research.google.com/github/maisiasr/resampling/blob/main/Resample_point_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Mount your google drive directory, you don't need to do this if you use your local machine.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from shapely.geometry import LineString
import os
from pathlib import Path

# ========================= CONFIGURATION =========================
source_dir = '/content/drive/MyDrive/Turkmenistan/Data/D3_Depth_120626'   # ← Make sure this is correct
output_dir = os.path.join(source_dir, 'Resampled_2000')

spacing = 2000

os.makedirs(output_dir, exist_ok=True)

# Get all .txt files (case insensitive)
txt_files = [f for f in os.listdir(source_dir) if f.lower().endswith('.txt')]

print(f"Found {len(txt_files)} .txt files to process.\n")

# ========================= BULK PROCESSING =========================
for filename in txt_files:
    try:
        file_path = os.path.join(source_dir, filename)
        print(f"Processing: {filename}")

        df = pd.read_csv(file_path, sep=r'\s+', engine='python')
        df.columns = df.columns.str.strip()

        if len(df.columns) < 2:
            print(f"  ⚠️ Skipping {filename}: Not enough columns")
            continue

        xcol = df.columns[0]
        ycol = df.columns[1]
        zcol = df.columns[2]

        if xcol not in df.columns or ycol not in df.columns or zcol not in df.columns:
            print(f"  ⚠️ Skipping {filename}: Missing X and/or Y and/or Z columns")
            continue

        line = LineString(zip(df[xcol], df[ycol], df[zcol]))

        if line.length < 1:
            print(f"  ⚠️ Skipping {filename}: Zero length")
            continue

        distances = np.arange(0, line.length, spacing)
        pts = [line.interpolate(d) for d in distances]

        out = pd.DataFrame({
            'X': [p.x for p in pts],
            'Y': [p.y for p in pts],
            'Z': [p.z for p in pts]
        })

        output_filename = filename.replace('.txt', f'_resampled_{spacing}.asc')
        output_path = os.path.join(output_dir, output_filename)

        out.to_csv(output_path, sep='\t', index=False)

        print(f"  ✅ Success → {len(out)} points (Original: {len(df)})")

    except Exception as e:
        print(f"  ❌ Error on {filename}: {e}")

print("\n🎉 Bulk processing finished!")
print(f"Output folder: {output_dir}")

Found 15 .txt files to process.

Processing: SR9_Cretaceous_Depth.txt
  ✅ Success → 54 points (Original: 4114)
Processing: SR7_S7_0SB_TopMaykop_Depth.txt
  ✅ Success → 1388 points (Original: 26202)
Processing: SR6_S6_0SB_TopDiatomite_Depth.txt
  ✅ Success → 2174 points (Original: 32211)
Processing: SR5_Shale_Diapir_Depth.txt
  ✅ Success → 477 points (Original: 8204)
Processing: SR5_S5SB_RS9_Depth.txt
  ✅ Success → 1652 points (Original: 31473)
Processing: SR4_6_S4_6SB_RS8B_Depth.txt
  ✅ Success → 1654 points (Original: 30704)
Processing: SR4_2_Top_Pre-Kinematic_Depth.txt
  ✅ Success → 1710 points (Original: 30457)
Processing: SR4_1_Intra_Syn-Kinematic_Depth.txt
  ✅ Success → 2216 points (Original: 32604)
Processing: SR4_0_Near_S4_0SB_NearRS2_Depth.txt
  ✅ Success → 2303 points (Original: 32604)
Processing: SR3_0_S3_0SB_Depth.txt
  ✅ Success → 2303 points (Original: 32604)
Processing: SR2_S2_0SB_RS1_Depth.txt
  ✅ Success → 2303 points (Original: 32604)
Processing: SR1_3_S1_3SB_Depth.txt